In [20]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/chethuhn/network-intrusion-dataset/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
/kaggle/input/datasets/chethuhn/network-intrusion-dataset/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
/kaggle/input/datasets/chethuhn/network-intrusion-dataset/Tuesday-WorkingHours.pcap_ISCX.csv
/kaggle/input/datasets/chethuhn/network-intrusion-dataset/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
/kaggle/input/datasets/chethuhn/network-intrusion-dataset/Monday-WorkingHours.pcap_ISCX.csv
/kaggle/input/datasets/chethuhn/network-intrusion-dataset/Friday-WorkingHours-Morning.pcap_ISCX.csv
/kaggle/input/datasets/chethuhn/network-intrusion-dataset/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
/kaggle/input/datasets/chethuhn/network-intrusion-dataset/Wednesday-workingHours.pcap_ISCX.csv


In [21]:

path = '/kaggle/input/datasets/chethuhn/network-intrusion-dataset'

csv_files = [
    os.path.join(path, f)
    for f in os.listdir(path)
    if f.endswith('.csv')
]
dfs = [pd.read_csv(f) for f in csv_files]
df = pd.concat(dfs, ignore_index=True)

KeyboardInterrupt: 

# **Data Preprocessing**

In [ ]:
df = df.replace([np.inf, -np.inf], np.nan)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler=MinMaxScaler()
df= df.dropna()
df.columns = df.columns.str.strip()

X = df.drop('Label', axis=1)
y = df['Label']

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
X_scaled = pd.DataFrame(
    X_scaled,
    columns=X.columns
)
X_scaled.head()

# **LSTM LAYER**

In [ ]:
from sklearn.model_selection import train_test_split
import tensorflow as tf

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train = X_train.to_numpy()
X_test = X_test.to_numpy()

X_train = X_train.astype(np.float32)  # add this
X_test = X_test.astype(np.float32)    # add this

timesteps = 10
features = X_train.shape[1]

def make_streaming_dataset(X, timesteps, batch_size=128, shuffle=False):
    n = len(X) - timesteps

    def generator():
        indices = np.arange(n)
        if shuffle:
            np.random.shuffle(indices)
        for i in indices:
            seq = X[i : i + timesteps]
            yield seq, seq

    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(timesteps, X.shape[1]), dtype=tf.float32),
            tf.TensorSpec(shape=(timesteps, X.shape[1]), dtype=tf.float32),
        )
    )
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE).repeat()

train_ds = make_streaming_dataset(X_train, timesteps, batch_size=128, shuffle=True)
val_ds   = make_streaming_dataset(X_test,  timesteps, batch_size=128, shuffle=False)


In [ ]:
from tensorflow.keras.layers import Input, LSTM, Dense, RepeatVector, TimeDistributed
from tensorflow.keras.models import Model

inputs = Input(shape=(timesteps, features))

# LSTM layer 1 (196 units)
x = LSTM(196, return_sequences=True, activation="relu")(inputs)

# LSTM layer 2 (125 units)
x = LSTM(125, return_sequences=False, activation="relu")(x)

# AE bottleneck (1 dense layer)
latent = Dense(16, activation="relu")(x)

In [23]:
x = RepeatVector(timesteps)(latent)

# mirror LSTM layers
x = LSTM(125, return_sequences=True, activation="relu")(x)
x = LSTM(196, return_sequences=True, activation="relu")(x)

outputs = TimeDistributed(Dense(features))(x)

from tensorflow.keras.optimizers import Adam
optimizer = Adam(learning_rate=0.001)
lstm_ae = Model(inputs, outputs)
lstm_ae.compile(optimizer=optimizer, loss="mae")

In [24]:
train_ds = make_streaming_dataset(X_train, timesteps, batch_size=128, shuffle=True)
val_ds   = make_streaming_dataset(X_test,  timesteps, batch_size=128, shuffle=False)

steps = (len(X_train) - timesteps) // 128
val_steps = (len(X_test) - timesteps) // 128

lstm_ae.fit(
    train_ds,
    epochs=10,
    steps_per_epoch=steps,
    validation_data=val_ds,
    validation_steps=val_steps,
)

Epoch 1/10
17674/17674 ━━━━━━━━━━━━━━━━━━━━ 2487s 140ms/step - loss: 0.0305 - val_loss: 0.0259
Epoch 2/10
17674/17674 ━━━━━━━━━━━━━━━━━━━━ 2461s 139ms/step - loss: 0.0234 - val_loss: 0.0214
Epoch 3/10
17674/17674 ━━━━━━━━━━━━━━━━━━━━ 2458s 139ms/step - loss: 0.0205 - val_loss: 0.0198
Epoch 4/10
17674/17674 ━━━━━━━━━━━━━━━━━━━━ 2475s 140ms/step - loss: 0.0194 - val_loss: 0.0190
Epoch 5/10
17674/17674 ━━━━━━━━━━━━━━━━━━━━ 2469s 140ms/step - loss: 0.0188 - val_loss: 0.0186
Epoch 6/10
17674/17674 ━━━━━━━━━━━━━━━━━━━━ 2480s 140ms/step - loss: 0.0184 - val_loss: 0.0183
Epoch 7/10
17674/17674 ━━━━━━━━━━━━━━━━━━━━ 2481s 140ms/step - loss: 0.0181 - val_loss: 0.0177
Epoch 8/10
17674/17674 ━━━━━━━━━━━━━━━━━━━━ 2481s 140ms/step - loss: 0.0176 - val_loss: 0.0174
Epoch 9/10
17674/17674 ━━━━━━━━━━━━━━━━━━━━ 2476s 140ms/step - loss: 0.0173 - val_loss: 0.0171
Epoch 10/10
17674/17674 ━━━━━━━━━━━━━━━━━━━━ 2477s 140ms/step - loss: 0.0171 - val_loss: 0.0170


In [36]:
n = len(X_test) - timesteps
X_test_seq = np.array([X_test[i:i+timesteps] for i in range(n)], dtype=np.float32)

X_pred = lstm_ae.predict(X_test_seq)
error = np.mean(np.abs(X_test_seq - X_pred), axis=(1,2))
threshold = np.percentile(error, 95) 
predictions = (error > threshold).astype(int)

17674/17674 ━━━━━━━━━━━━━━━━━━━━ 315s 18ms/step


In [37]:
from sklearn.metrics import classification_report, confusion_matrix

y_test_seq = y_test.to_numpy()[timesteps:]
y_binary = (y_test_seq != 'BENIGN').astype(int)

print(confusion_matrix(y_binary, predictions))
print(classification_report(y_binary, predictions))

[[431392  22865]
 [105895   5414]]
              precision    recall  f1-score   support

           0       0.80      0.95      0.87    454257
           1       0.19      0.05      0.08    111309

    accuracy                           0.77    565566
   macro avg       0.50      0.50      0.47    565566
weighted avg       0.68      0.77      0.71    565566



In [38]:
for p in [50, 55, 60, 65, 70, 75, 80, 85, 90, 95]:
    thresh = np.percentile(error, p)
    preds = (error > thresh).astype(int)
    tp = ((preds == 1) & (y_binary == 1)).sum()
    fp = ((preds == 1) & (y_binary == 0)).sum()
    fn = ((preds == 0) & (y_binary == 1)).sum()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    print(f"p={p}: precision={precision:.2f}, recall={recall:.2f}, f1={f1:.2f}")

p=50: precision=0.20, recall=0.50, f1=0.28
p=55: precision=0.20, recall=0.45, f1=0.27
p=60: precision=0.20, recall=0.40, f1=0.26
p=65: precision=0.20, recall=0.35, f1=0.25
p=70: precision=0.20, recall=0.30, f1=0.24
p=75: precision=0.20, recall=0.25, f1=0.22
p=80: precision=0.20, recall=0.20, f1=0.20
p=85: precision=0.20, recall=0.15, f1=0.17
p=90: precision=0.20, recall=0.10, f1=0.13
p=95: precision=0.19, recall=0.05, f1=0.08
